# AnimalPoseTracker Demo

Run a 10-epoch AnimalRTPose-N workflow on Colab: training, validation, test prediction from `.pt`, ONNX export, and test prediction from ONNX. The dataset stays in `MyDrive/datasets/trimouse`; the repository and project stay under `MyDrive/wux024`. Enable a GPU runtime before running.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/wux024/AnimalPoseTracker.git"
BRANCH = "preview"
DATA_SOURCE_ROOT = Path("/content/drive/MyDrive/datasets/trimouse").expanduser().resolve()
WORKSPACE_ROOT = Path("/content/drive/MyDrive/wux024")
REPO_DIR = WORKSPACE_ROOT / "AnimalPoseTracker"
PROJECTS_ROOT = WORKSPACE_ROOT
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")

EPOCHS = 10
BATCH_SIZE = 4
IMAGE_SIZE = 640
NUM_WORKERS = 2
DEVICE = "0"
PREDICT_SPLIT = "test"

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print("Repository:", REPO_DIR, REPO_COMMIT)
print("Dataset:", DATA_SOURCE_ROOT)

In [ ]:
%cd /content/drive/MyDrive/wux024/AnimalPoseTracker
%pip install -v --no-deps -e .
%pip install -q pycocotools onnx onnxsim onnxruntime

In [ ]:
import yaml

if not DATA_SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f"Trimouse dataset not found: {DATA_SOURCE_ROOT}")
for split in ("train", "val", "test"):
    if not (DATA_SOURCE_ROOT / "images" / split).is_dir():
        raise FileNotFoundError(f"Missing image split: {DATA_SOURCE_ROOT / 'images' / split}")

DATA_YAML = yaml.safe_load(
    (REPO_DIR / "animalposetracker" / "cfg" / "datasets" / "trimouse.yaml")
    .read_text(encoding="utf-8")
)
DATA_YAML["path"] = str(DATA_SOURCE_ROOT)
if not DATA_YAML.get("annotation_format"):
    annotation_dir = DATA_SOURCE_ROOT / "annotations"
    annotation_files = [
        annotation_dir / f"{split}.json" for split in ("train", "val", "test")
    ] + [
        annotation_dir / f"person_keypoints_{split}2017.json"
        for split in ("train", "val", "test")
    ]
    DATA_YAML["annotation_format"] = (
        "coco" if any(path.is_file() for path in annotation_files) else "yolo"
    )
print("Annotation format:", DATA_YAML["annotation_format"])
print("Keypoint shape:", DATA_YAML["kpt_shape"])

In [ ]:
from animalposetracker.project import AnimalPoseTrackerProject

keypoint_count, keypoint_dims = DATA_YAML["kpt_shape"]
class_names = {int(key): str(value) for key, value in DATA_YAML["names"].items()}
project = AnimalPoseTrackerProject(
    local_path=PROJECTS_ROOT,
    project_name="trimouse",
    worker="colab",
    model_type="AnimalRTPose",
    model_scale="N",
    keypoints=keypoint_count,
    visible=(keypoint_dims == 3),
    classes=len(class_names),
    keypoints_name=DATA_YAML.get(
        "keypoint_names", [f"kpt_{index}" for index in range(keypoint_count)]
    ),
    skeleton=DATA_YAML.get("skeleton", []),
    kpt_oks_sigmas=DATA_YAML.get("kpt_oks_sigmas"),
    classes_name=[class_names[index] for index in sorted(class_names)],
    date=RUN_ID,
)
project.create_new_project()
project.dataset_config.update(DATA_YAML)
project.dataset_config["path"] = str(DATA_SOURCE_ROOT)
project.dataset_config["names"] = class_names
project.project_config.update({
    "classes": len(class_names),
    "classes_name": [class_names[index] for index in sorted(class_names)],
    "keypoints": keypoint_count,
    "keypoints_name": DATA_YAML.get(
        "keypoint_names", [f"kpt_{index}" for index in range(keypoint_count)]
    ),
    "visible": (keypoint_dims == 3),
    "skeleton": DATA_YAML.get("skeleton", []),
    "project_path": str(project.project_path),
})
project._sync_model_config_from_dataset()
project.other_config.update({
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "imgsz": IMAGE_SIZE,
    "workers": NUM_WORKERS,
    "device": DEVICE,
    "pretrained": True,
    "val": True,
    "plots": True,
    "project": "runs",
    "name": "train",
    "exist_ok": False,
})
project._detect_pretrained()
project.save_configs("all")
PROJECT_DIR = project.project_path
print("Project:", PROJECT_DIR)
print("Pretrained weights:", project.other_config.get("pretrained"))

In [ ]:
import json
import os
import shlex
import subprocess
from pathlib import Path


def _format_cli_metrics(metrics):
    return ", ".join(
        f"{key}={value:.4g}"
        for key, value in metrics.items()
        if isinstance(value, (int, float))
    )


def _show_cli_line(line):
    text = line.rstrip()
    if not text:
        return
    try:
        event = json.loads(text)
    except json.JSONDecodeError:
        print(text, flush=True)
        return
    if not isinstance(event, dict) or "event" not in event:
        print(text, flush=True)
        return

    kind = event["event"]
    epoch = event.get("epoch")
    epochs = event.get("epochs")
    metrics = _format_cli_metrics(event.get("metrics") or {})
    if kind == "batch":
        position = f"epoch {epoch}/{epochs}, batch {event.get('step')}/{event.get('steps')}"
        print(f"  {position}" + (f" | {metrics}" if metrics else ""), flush=True)
    elif kind == "epoch_end":
        print(f"Epoch {epoch}/{epochs} complete" + (f" | {metrics}" if metrics else ""), flush=True)
    elif kind == "validation":
        print(f"Validation {epoch}/{epochs}" + (f" | {metrics}" if metrics else ""), flush=True)
    else:
        details = [event.get("message"), metrics]
        details = [str(value) for value in details if value]
        print(f"[{kind}] " + " | ".join(details), flush=True)


def run_cli(*arguments):
    command = ["animalpose-cli", *map(str, arguments)]
    print("$ " + " ".join(shlex.quote(value) for value in command), flush=True)
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Could not read CLI output")
    for line in process.stdout:
        _show_cli_line(line)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


os.chdir(PROJECT_DIR)
print("Starting training. Progress events will appear below.", flush=True)
run_cli("train", "--config", "configs/other.yaml")
checkpoint_candidates = sorted(
    Path("runs").glob("train*/weights/best.pt"),
    key=lambda path: path.stat().st_mtime,
)
if not checkpoint_candidates:
    raise FileNotFoundError("Training did not produce runs/train*/weights/best.pt")
BEST_CHECKPOINT = checkpoint_candidates[-1].resolve()
print("Best checkpoint:", BEST_CHECKPOINT)


In [ ]:
import json
from pathlib import Path

run_cli("val", "--config", "configs/other.yaml", "--weights", BEST_CHECKPOINT)
metric_candidates = sorted(
    Path("runs").glob("val*/metrics.json"),
    key=lambda path: path.stat().st_mtime,
)
if not metric_candidates:
    raise FileNotFoundError("Validation did not produce runs/val*/metrics.json")
VALIDATION_METRICS = metric_candidates[-1].resolve()
print("Validation metrics:", json.dumps(
    json.loads(VALIDATION_METRICS.read_text(encoding="utf-8")),
    indent=2,
    ensure_ascii=False,
))


In [ ]:
from pathlib import Path

run_cli(
    "predict", "--config", "configs/other.yaml",
    "--weights", BEST_CHECKPOINT, "--split", PREDICT_SPLIT,
)
prediction_candidates = sorted(
    Path("runs").glob("predict*/predictions.json"),
    key=lambda path: path.stat().st_mtime,
)
if not prediction_candidates:
    raise FileNotFoundError("Prediction did not produce runs/predict*/predictions.json")
PT_PREDICTIONS = prediction_candidates[-1].resolve()
print(".pt predictions:", PT_PREDICTIONS)


In [ ]:
from pathlib import Path

run_cli(
    "export", "--config", "configs/other.yaml",
    "--weights", BEST_CHECKPOINT, "--format", "onnx",
    "--output-dir", "runs/export",
)
onnx_candidates = sorted(
    Path("runs").glob("export*/best.onnx"),
    key=lambda path: path.stat().st_mtime,
)
if not onnx_candidates:
    raise FileNotFoundError("Export did not produce runs/export*/best.onnx")
ONNX_MODEL = onnx_candidates[-1].resolve()
run_cli(
    "predict", "--config", "configs/other.yaml",
    "--weights", ONNX_MODEL, "--split", PREDICT_SPLIT,
)
onnx_prediction_candidates = sorted(
    Path("runs").glob("predict*/predictions.json"),
    key=lambda path: path.stat().st_mtime,
)
if not onnx_prediction_candidates:
    raise FileNotFoundError("ONNX prediction did not produce runs/predict*/predictions.json")
ONNX_PREDICTIONS = onnx_prediction_candidates[-1].resolve()
print("ONNX model:", ONNX_MODEL)
print("ONNX predictions:", ONNX_PREDICTIONS)


In [ ]:
import json
from pathlib import Path
import yaml

project_path = Path(PROJECT_DIR).resolve()

def make_pretrained_paths_relative(value):
    if isinstance(value, dict):
        for key, item in list(value.items()):
            if key == "pretrained" and isinstance(item, str):
                candidate = Path(item).expanduser()
                if candidate.is_absolute():
                    try:
                        value[key] = str(candidate.resolve().relative_to(project_path))
                    except ValueError:
                        pass
            else:
                make_pretrained_paths_relative(item)
    elif isinstance(value, list):
        for item in value:
            make_pretrained_paths_relative(item)


dataset_path = project_path / "configs" / "dataset.yaml"
dataset_config = yaml.safe_load(dataset_path.read_text(encoding="utf-8"))
dataset_config["path"] = str(DATA_SOURCE_ROOT)
dataset_path.write_text(
    yaml.safe_dump(dataset_config, sort_keys=False, allow_unicode=True), encoding="utf-8"
)
other_path = project_path / "configs" / "other.yaml"
other_config = yaml.safe_load(other_path.read_text(encoding="utf-8"))
other_config["model"] = "configs/model.yaml"
other_config["data"] = "configs/dataset.yaml"
make_pretrained_paths_relative(other_config)
other_path.write_text(
    yaml.safe_dump(other_config, sort_keys=False, allow_unicode=True), encoding="utf-8"
)

def relative(path):
    try:
        return str(Path(path).resolve().relative_to(project_path))
    except ValueError:
        return str(path)

summary = {
    "repository": REPO_URL,
    "branch": BRANCH,
    "commit": REPO_COMMIT,
    "model": "AnimalRTPose-N",
    "dataset": str(DATA_SOURCE_ROOT),
    "annotation_format": DATA_YAML["annotation_format"],
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "best_checkpoint": relative(BEST_CHECKPOINT),
    "validation_metrics": relative(VALIDATION_METRICS),
    "pt_predictions": relative(PT_PREDICTIONS),
    "onnx_model": relative(ONNX_MODEL),
    "onnx_predictions": relative(ONNX_PREDICTIONS),
    "project": str(project_path),
}
summary_path = project_path / "notebook_run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + chr(10), encoding="utf-8")
print("Saved project and outputs:", project_path)
print(json.dumps(summary, indent=2, ensure_ascii=False))